In [1]:
# 環境架設
from selenium import webdriver
from selenium.webdriver.remote.webelement import WebElement
# driver = webdriver.Chrome()
# driver = webdriver.Firefox()
from selenium.webdriver.chrome.service import Service as ChromeService
# from webdriver_manager.chrome import ChromeDriverManager
# pip install webdriver-manager
# driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))

import xpon_module as xpon
import selenium_module as mod
import xpon_olt_status_module as olts
import xpon_olt_config_module as oltc
import xpon_batch_ont_config_module as bat
import xpon_ont_config_module as ontc
import xpon_profiles_module as pro
# import multicast_module as mul

# copy from GLC project-------------
import acl_module as acl  
import glc_test_module as glc
# ----------------------------------

from selenium.webdriver.support.ui import Select
import time
import datetime
import pyautogui
# from docx import Document
# from docx.shared import Inches

import ems_acl_module as e_acl
import ems_profile_module as e_pro
import ems_test_module as ems
import ems_test_module_OLT1408 as emss
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from customdriver import CustomDriver
from variable_module import *
import pyautogui
import get_card
# import videocap_module as videocap

time_sleep_before_find_element = 0 # 0.1
driver = CustomDriver(time_sleep = time_sleep_before_find_element)
try:
    driver.maximize_window()    
except:
    pass


start_time = time.time()
start_time0 = time.time()



ModuleNotFoundError: No module named 'pyrsistent'

In [ ]:
from variable_module import *
# RESET from Telnet 
import time
import datetime
import telnetlib
 
def Input_cmd_func(input_cmd_file_name):
    # Get command from txt
    f = open(input_cmd_file_name,'r', encoding="utf-8")
    My_List = []
    done = 0
    while (not done):
        aline = f.readline()
        if(aline!=''):
            My_List.append(aline.strip())
        else:
            done = 1
    return My_List

def Telnet_func(host_ip,My_List,output_file, _type=0):	

    print(f"\n= = = = = = = = start = = = = = = = = = = = = = = = =\n",file = open(output_file, '+a'))  

    def wait_for_specific_words(tn,words):
        while True:
            output = tn.read_until(b"\n").decode("utf-8")
            output = str(output)
            for word in words:
                if(word.decode("utf-8") in output):
                    break
    wordds = [b"# ",b"-- more --, next page: Space, continue: c, quit: ESC"]


    username = 'admin'
    password = '1234'
    print(f'### telnet {host_ip}',file = open(output_file, '+a'))
    tnet_hndl = telnetlib.Telnet(host_ip)
    if(_type==1):
        print(tnet_hndl.read_until(b"User name: "),file = open(output_file, '+a'))
        tnet_hndl.write(username.encode('ascii') + b"\n")
    else:
        print(tnet_hndl.read_until(b"login: "),file = open(output_file, '+a')) # for IES52xx
        tnet_hndl.write(username.encode('ascii') + b"\n")
    print(tnet_hndl.read_until(b"Password: "),file = open(output_file, '+a'))
    tnet_hndl.write(password.encode('ascii') + b"\n")
    print(tnet_hndl.read_until(b"# "),file = open(output_file, '+a'))
    for cmd in My_List:


        if((cmd=="c")or(cmd=="y")):
            cmd = bytes(cmd, encoding='UTF-8')
            tnet_hndl.write(cmd)
        else:
            cmd = bytes(cmd, encoding='UTF-8')
            tnet_hndl.write(cmd + b"\n")
        print(cmd,file = open(output_file, '+a'))  # print command

        time.sleep(0) # 1 # 0.3

        if_find = False
        for i in range(3):
            set_timeout = 5
            if(i>0):
                set_timeout = 20
            for wor in wordds:
                # print(f'start read until {wor}')
                try:
                    # list_1 = str(tnet_hndl.read_until(wor,timeout=10)).split('\\r\\n')
                    list_1 = str(tnet_hndl.read_until(wor,timeout=set_timeout)).split('\\r\\n')
                    if_find = True
                    # print(f'read until {wor} pass')
                    break
                except ConnectionResetError as e:
                    print(f'{cmd} //read until {wor} fail ({i+1}t)')
                    print('timeout occurred')
                    continue
            if(if_find == True):
                break
        if(if_find == False):
            print(f"'{cmd}' //read until all words fail",file = open(output_file, '+a'))

        list_1.pop(0) ### delete the first line
        for i in list_1:
            print(f':{i}',file = open(output_file, '+a'))  # print return data
        print('\n',file = open(output_file, '+a'))
    tnet_hndl.close()

    print(f"\n= = = = = = = = done. = = = = = = = = = = = = = = = =\n",file = open(output_file, '+a'))  

My_List = ['er ru ext','y']
output_file = 'temp_erase_log.txt'
output_file_f = open(output_file,"w+")
Telnet_func(url_ip,My_List,output_file, _type=1)
time.sleep(10)

In [ ]:
#   Login
driver.get(ems_url)
mod.Login_EMS(driver,username,ems_password)
time.sleep(5)

# Get EMS version
try:
    driver.find_element('id','helpMenu').click()
except:
    mod.Login_EMS(driver,username,ems_password)
    time.sleep(5)
    driver.find_element('id','helpMenu').click()
driver.find_element('id','about_ems_button').click()
time.sleep(3)
emsversion = driver.find_element('id','emsversion').text.split("\n")[0]
time.sleep(5)
driver.find_element('xpath', '//*[@type="button" and @class="button button_green"]').click()    ### click close

# driver.get("http://192.168.169.207")
# # may need click the browser
# time.sleep(2)
# pyautogui.write(username)
# pyautogui.press("tab")
# pyautogui.write(password)
# pyautogui.press("enter")
driver.get(f"http://{username}:{password}@{url_ip}")
time.sleep(5)

#   Get Web Info
mod.Switch_Default(driver)
mod.Switch_Specific_Frame(driver,3)
driver.find_element(By.LINK_TEXT,"Basic Setting").click()
time.sleep(1)
mod.Switch_Default(driver)
mod.Switch_Specific_Frame(driver,4)
driver.find_element(By.LINK_TEXT,"System Info").click()
mod.Switch_Default(driver)
mod.Switch_Specific_Frame(driver,6)
time.sleep(1)
model_name = driver.find_element(By.XPATH,"/html/body/form/table[5]/tbody/tr[3]/td[2]").text
model_version = driver.find_element(By.XPATH,"/html/body/form/table[5]/tbody/tr[5]/td[2]").text.split("|")[0]
#    Set File Name
date1 = str(datetime.datetime.now()).split(":")[0]
date2 = str(datetime.datetime.now()).split(":")[1]
# file_name = f"Output\Web_{model_name}_{model_version}_{date1}{date2}.txt"
file_name = f"Output\Web_EMS_{model_name}_{emsversion}_{date1}{date2}.txt"

#    Info in File
test_report = open(file_name,"w+")
print("Web Auto Report",file = open(file_name, '+a'))
print(datetime.datetime.now(),file = open(file_name, '+a'))
print(f"EMS Web Version: {emsversion}\nFW Web IP:{url}\nFW Web model: {model_name}\nFW Web version: {model_version}",file = open(file_name, '+a'))

#    Set Logout Time
mod.Switch_Default(driver)
mod.Switch_Specific_Frame(driver,3)
driver.find_element(By.LINK_TEXT,"Management").click()
time.sleep(1)
mod.Switch_Default(driver)
mod.Switch_Specific_Frame(driver,4)
driver.find_element(By.LINK_TEXT,"Access Control").click()
mod.Switch_Default(driver)
mod.Switch_Specific_Frame(driver,6)
time.sleep(1)
driver.find_element(By.XPATH,'/html/body/table/tbody/tr[2]/td/table/tbody/tr[6]/td[3]/a').click()
time.sleep(2)
driver.find_element(By.NAME,"RpAccessSv_IptHttpTimeout").send_keys(Keys.CONTROL + 'a')
driver.find_element(By.NAME,"RpAccessSv_IptHttpTimeout").send_keys('255')
driver.find_element(By.NAME,'RpAccessSv_BtnApply').click()
time.sleep(2)



In [ ]:
if(model_name=="OLT1404A-IA"):
    a=int(OLT_Eth_Port)-4
    Pon_Name="PON "+OLT_Port
    Pon_Name_L="pon-"+OLT_Port
    Eth_Name="ETH "+str(a)
    Eth_Name_L="eth-"+str(a)
    print(Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)
elif(model_name=="OLT1404B-IA"):
    a=int(OLT_Eth_Port)-4
    Pon_Name="PON "+OLT_Port
    Pon_Name_L="pon-"+OLT_Port
    Eth_Name="ETH "+str(a)
    Eth_Name_L="eth-"+str(a)
    print(Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)
elif(model_name=="OLT1408A-IA"):
    a=int(OLT_Eth_Port)-8
    Pon_Name="PON "+OLT_Port
    Pon_Name_L="pon-"+OLT_Port
    Eth_Name="ETH "+str(a)
    Eth_Name_L="eth-"+str(a)
    print(Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)
elif(model_name=="OLT1408B-IA"):
    a=int(OLT_Eth_Port)-8
    Pon_Name="PON "+OLT_Port
    Pon_Name_L="pon-"+OLT_Port
    Eth_Name="ETH "+str(a)
    Eth_Name_L="eth-"+str(a)
    print(Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)
elif(model_name=="OLT1408A-C"):
    a=int(OLT_Eth_Port)-8
    Pon_Name="PON "+OLT_Port
    Pon_Name_L="pon-"+OLT_Port
    Eth_Name="ETH "+str(a)
    Eth_Name_L="eth-"+str(a)

In [ ]:
print(f"EMS - Configuration for OLT140x Test Start",file = open(file_name, '+a'))
driver.get(ems_url)
mod.Login_EMS(driver,username,ems_password)
time.sleep(5)

#ems.EMS_Provision_Config_path(driver,url)

In [ ]:
# from ems_test_module_OLT1408 import *
# driver.get(ems_url)
# EMS_Provision_Config_path(driver,url,file_name)
# driver.find_element_by_link_text("Advanced Application").click() 
# time.sleep(1)
# driver.find_element_by_link_text("PPPoE").click()
# time.sleep(1)
# driver.find_element_by_link_text("Advanced Application").click()
# time.sleep(1)
# driver.find_element(By.ID,"advance_pppoe_vlan_profile_new_btn").click()
# time.sleep(3)
# e_swi.Advanced_PPPoE_VLAN(driver,"4094","4094","Yes",True,True,file_name)
# msg = e_swi.alert_accept_text(driver)
# try:
#     e_pro.Close(driver)
# except:
#     pass

# #[EMS1-6577][Advanced - PPPoE VLAN - New] Result check

# driver.get(url)
# time.sleep(2)
# mod.Switch_Default(driver)
# mod.Switch_Specific_Frame(driver,3)
# driver.find_element(By.LINK_TEXT,"Advanced Application").click()
# time.sleep(1)
# mod.Switch_Default(driver)
# mod.Switch_Specific_Frame(driver,4)
# driver.find_element(By.LINK_TEXT,"PPPoE").click()
# time.sleep(1)
# mod.Switch_Default(driver)
# mod.Switch_Specific_Frame(driver,6)
# driver.find_element(By.LINK_TEXT,"Click here").click()
# time.sleep(1)
# driver.find_element(By.LINK_TEXT,"VLAN").click()
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Ipt_StartVlan").send_keys(e_swi.Keys.CONTROL + 'a')
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Ipt_StartVlan").send_keys("4094")
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Ipt_EndVlan").send_keys(e_swi.Keys.CONTROL + 'a')
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Ipt_EndVlan").send_keys("4094")
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Btn_Apply1").click()
# time.sleep(5)
# if_temp_error = False
# if_find = False
# for i in range(1,driver.find_elements(By.NAME,"rpPPPoEIAConfV_Chk_RemoteID").__len__()+2):
#     ele = driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[2]/div").text
#     if(ele == '4094'):
#         if(Select(driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[3]/div/select")).first_selected_option.get_attribute("text") == "Yes"):
#             print(f"             [EMS1-6577][Advanced - PPPoE VLAN - New] Enable (Yes) Pass",file = open(file_name, '+a'))
#         else:
#             print(f'             [EMS1-6577][Advanced - PPPoE VLAN - New] Enable ({Select(driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[3]/div/select")).first_selected_option.get_attribute("text")} != Yes) Fail',file = open(file_name, '+a'))
#             if_temp_error = True

#         if(driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[4]/div/input").is_selected() == True):
#             print(f"             [EMS1-6577][Advanced - PPPoE VLAN - New] Circuit_id (True) Pass",file = open(file_name, '+a'))
#         else:
#             print(f'             [EMS1-6577][Advanced - PPPoE VLAN - New] Circuit_id ({driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[4]/div/input").is_selected()} != True) Fail',file = open(file_name, '+a'))
#             if_temp_error = True

#         if(driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[5]/div/input").is_selected() == True):
#             print(f"             [EMS1-6577][Advanced - PPPoE VLAN - New] Remote_id (True) Pass",file = open(file_name, '+a'))
#         else:
#             print(f'             [EMS1-6577][Advanced - PPPoE VLAN - New] Remote_id ({driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[5]/div/input").is_selected()} != True) Fail',file = open(file_name, '+a'))
#             if_temp_error = True
#         if_find = True
# if if_find == False:
#     print(f"             [EMS1-6577][Advanced - PPPoE VLAN - New] 4094 Not Found Test Fail",file = open(file_name, '+a'))
#     if_temp_error = True

# if(if_temp_error==False):
#     print(f"             [EMS1-6577][Advanced - PPPoE VLAN - New] Result Pass",file = open(file_name, '+a'))
# else:
#     print(f"             [EMS1-6577][Advanced - PPPoE VLAN - New] Result Fail",file = open(file_name, '+a'))
        

# #[EMS1-6578][Advanced - PPPoE VLAN - Modify] Result

# driver.get(ems_url)
# EMS_Provision_Config_path(driver,url,file_name)
# driver.find_element_by_link_text("Advanced Application").click() 
# time.sleep(1)
# driver.find_element_by_link_text("PPPoE").click()
# time.sleep(1)
# driver.find_element_by_link_text("Advanced Application").click()
# time.sleep(1)

# if_find = False
# for i in range(1,int(driver.find_element(By.ID,"pageTotal2").text)+1):
#     for j in range(1,driver.find_elements(By.NAME,"advance_pppoe_vlan_profile_checkbox").__len__()+1):
#         ele = driver.find_element(By.XPATH,f"/html/body/div[3]/div[2]/div/div/div/div[14]/div[2]/div[1]/div[{j}]/div[2]").text
#         if(ele == '4094'):
#             driver.find_element(By.XPATH,f"/html/body/div[3]/div[2]/div/div/div/div[14]/div[2]/div[1]/div[{j}]/div[6]/label").click()
#             if_find =True
#             time.sleep(1)
#         if(if_find==True):
#             break 
#     if(if_find==True):
#         break
#     else:
#         driver.find_element(By.ID,"btnNextPage2").click()

# driver.find_element(By.ID,"advance_pppoe_vlan_profile_mod_btn").click()
# e_swi.Advanced_PPPoE_VLAN_Modify(driver,"Yes",False,False,file_name)

# #[EMS1-6578][Advanced - PPPoE VLAN - Modify] Result check

# driver.get(url)
# time.sleep(2)
# mod.Switch_Default(driver)
# mod.Switch_Specific_Frame(driver,3)
# driver.find_element(By.LINK_TEXT,"Advanced Application").click()
# time.sleep(1)
# mod.Switch_Default(driver)
# mod.Switch_Specific_Frame(driver,4)
# driver.find_element(By.LINK_TEXT,"PPPoE").click()
# time.sleep(1)
# mod.Switch_Default(driver)
# mod.Switch_Specific_Frame(driver,6)
# driver.find_element(By.LINK_TEXT,"Click here").click()
# time.sleep(1)
# driver.find_element(By.LINK_TEXT,"VLAN").click()
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Ipt_StartVlan").send_keys(e_swi.Keys.CONTROL + 'a')
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Ipt_StartVlan").send_keys("4094")
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Ipt_EndVlan").send_keys(e_swi.Keys.CONTROL + 'a')
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Ipt_EndVlan").send_keys("4094")
# driver.find_element(By.NAME,"rpPPPoEIAConfV_Btn_Apply1").click()
# time.sleep(3)
# if_temp_error = False
# if_find = False
# for i in range(1,driver.find_elements(By.NAME,"rpPPPoEIAConfV_Chk_RemoteID").__len__()+2):
#     ele = driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[2]/div").text
#     if(ele == '4094'):
#         if(Select(driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[3]/div/select")).first_selected_option.get_attribute("text") == "Yes"):
#             print(f"             [EMS1-6578][Advanced - PPPoE VLAN - Modify] Enable (Yes) Pass",file = open(file_name, '+a'))
#         else:
#             print(f'             [EMS1-6578][Advanced - PPPoE VLAN - Modify] Enable ({Select(driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[3]/div/select")).first_selected_option.get_attribute("text")} != Yes) Fail',file = open(file_name, '+a'))
#             if_temp_error = True

#         if(driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[4]/div/input").is_selected() == False):
#             print(f"             [EMS1-6578][Advanced - PPPoE VLAN - Modify] Circuit_id (False) Pass",file = open(file_name, '+a'))
#         else:
#             print(f'             [EMS1-6578][Advanced - PPPoE VLAN - Modify] Circuit_id ({driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[4]/div/input").is_selected()} != False) Fail',file = open(file_name, '+a'))
#             if_temp_error = True

#         if(driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[5]/div/input").is_selected() == False):
#             print(f"             [EMS1-6578][Advanced - PPPoE VLAN - Modify] Remote_id (False) Pass",file = open(file_name, '+a'))
#         else:
#             print(f'             [EMS1-6578][Advanced - PPPoE VLAN - Modify] Remote_id ({driver.find_element(By.XPATH,f"/html/body/form/table/tbody/tr[3]/td/table/tbody/tr/td/table[1]/tbody/tr[{i*2}]/td[5]/div/input").is_selected()} != False) Fail',file = open(file_name, '+a'))
#             if_temp_error = True
#         if_find = True
# if if_find == False:
#     print(f"             [EMS1-6578][Advanced - PPPoE VLAN - Modify] 4094 Not Found Test Fail",file = open(file_name, '+a')) 
#     if_temp_error = True
# if(if_temp_error==False):
#     print(f"             [EMS1-6578][Advanced - PPPoE VLAN - Modify] Result Pass",file = open(file_name, '+a'))
# else:
#     print(f"             [EMS1-6578][Advanced - PPPoE VLAN - Modify] Result Fail",file = open(file_name, '+a')) 

In [ ]:
emss.get_file_name(file_name)
emss.Basic_General(driver,file_name)  
emss.Basic_Switch(driver,file_name)                                                                                            #set fail   k
emss.Basic_IP(driver,file_name)                                                                                                    #set fail   k
emss.Basic_Port(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                          #OK         k
emss.IP_Static(driver,file_name)                                                                                                 #
emss.IP_DHCP(driver,file_name)                                                                                                     #           k
emss.Advanced_VLAN(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                       #VLAN - Protocol Based SET Fail k
emss.Advanced_Static(driver,file_name)                                                                                             #OK         k
emss.Advanced_Filtering(driver,file_name)                                                                                          #OK         k
emss.Advanced_Broadcast(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                 #OK         k
emss.Advanced_Link(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                       #OK         k
emss.Advanced_Port(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                       #ok         k
emss.Advanced_Classifier(driver,file_name)                                                                                         #EMS Ethernet Type無法修改
emss.Advanced_Policy(driver,file_name)                                                                                             #OK         k
emss.Advanced_Queqe(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                      #OK         k
emss.Advanced_VLANStacking(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                               #OK         k
emss.Advanced_Multicast(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                  #set fail   k
emss.Advanced_IP(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                         #OK         k
emss.Advanced_Loop(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                       #OK         k
emss.Advanced_PPPoE(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                      #OK         k
emss.GPON_OLT(driver,file_name,Pon_Name,Eth_Name,Pon_Name_L,Eth_Name_L)                                                            #           k
emss.Configure_Save(driver,file_name)                                                                                              #OK         k

In [ ]:
import ems_module_OLT1408 as e_swi
import web_share.EMS.ems_module as e_mod
from selenium.webdriver.common.keys import Keys
#  ems_test_module 424,1268 need fix
#  還沒寫 [EMS1-6534][VLAN - Subnet Based] Page Switch Result Pass/Fail
#       [EMS1-6447][Basic - IP Setup] Apply Result Pass/Fail
#       [EMS1-6538][VLAN - Protocol Based] Page Switch Result Pass/Fail
#       #VLAN - Static VLAN Delete Result  2889
#       1935 2615 Modify不會用 2025 2281 2683 Delete check不會 2970
#       EMS 296
#       3878 PON要判斷他是幾PON的去推算ETH8 在哪裡
#   General_port  select
#   4650 無法NEW EMS沒有名字空格 4747 Modify不知道抓哪個元素 Delete還沒寫 #[EMS1-6561][Advanced - Multicast] Modify Result check
#   [EMS1-6570][Advanced - IP Source Guard] VLAN Configure Table Apply Result Pass/Fail 5183 result wrong
#   [EMS1-6558][Advanced - Classifier] Detail Result 4112
#   5186 判斷是為何要+3?



In [ ]:
# #    FW Web EMS Web CardType Case       
# # ====================================

# driver.get(url) 
# mod.Login(driver,username,password) 
 
# # FW get model_name, version
# # ...


# # Config Reset ---
# # ems.Maintenance_Config_Reset(driver)  #*自動化全程都不能點其他東西，測試頁面必須在最前面

# # FW Sys - Access Control - SNMP, input value from ENV.txt ---
# # FW Sys - User Account, input value from ENV.txt ---
# ems.EMS_Set_Prepare(driver,snmp_get,snmp_set,snmp_version,username,password)

# # cardtype case
# list1 = get_card.get_card_case(driver,glc_port,pon_x_1_aid,pon_1_aid,file_name='')

# # kick?
# # driver.quit()
# ems.FW_Logout(driver)


# #    EMS Web        
# # ====================================
# # driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))
# # driver = CustomDriver(time_sleep = time_sleep_before_find_element)
# driver.get("https://192.168.128.8:9116/") 
# time.sleep(5)
# mod.Login_EMS(driver,username,ems_password)
# time.sleep(5)
# try:
#     driver.find_element('id','helpMenu').click()
# except:
#     mod.Login_EMS(driver,username,ems_password)
#     time.sleep(5)
#     driver.find_element('id','helpMenu').click()
# driver.find_element('id','about_ems_button').click()
# time.sleep(3)
# emsversion = driver.find_element('id','emsversion').text.split("\n")[0]
# time.sleep(5)
# driver.find_element('xpath', '//*[@type="button" and @class="button button_green"]').click()    ### click close

# #    Set File Name
# date1 = str(datetime.datetime.now()).split(":")[0]
# date2 = str(datetime.datetime.now()).split(":")[1]
# file_name = f"Output\Web_EMS_{emsversion}_{date1}{date2}.txt"
# # file_name = f"Output\Web_EMS_test.txt"

# #    Info in File
# test_report = open(file_name,"w+")
# print("EMS Web Auto Report",file = open(file_name, '+a'))
# print(datetime.datetime.now(),file = open(file_name, '+a'))
# print(f"EMS Web IP:http://192.168.128.8:9116/  \nEMS Version:{emsversion} ",file = open(file_name, '+a'))
# print(f"FW Web IP:{url}FW MSC:{msc_slot}-{msc_port}/{msc_card_type}/{msc_version}",file = open(file_name, '+a'))
# print(f"FW Web IP:{url}FW {msc_card_type}/{msc_version}",file = open(file_name, '+a'))
# print(f"{list1}",file = open(file_name, '+a'))
# get_card.print_case(list1,file_name)
# MSC_slot = list1.pop()
# MSC_card = list1.pop()
# print(f"  (find_element(), find_elements() will wait {time_sleep_before_find_element} sec )",file = open(file_name, '+a'))
# print("======================================================================================",file = open(file_name, '+a'))

# print(f"EMS Test\nTest Start",file = open(file_name, '+a')) 


In [ ]:
# # #    Provision > Profile 
# # # ====================================

# print(f"EMS - Profile Test Start\n",file = open(file_name, '+a'))

# ems.EMS_delete_profiles(driver,username,ems_password,file_name)
# ems.EMS_test_all_Profile(driver,url,list1,MSC_card,MSC_slot,username,ems_password,file_name,  2)



# # # ems.IGMP_Group_Privilege_Profile(driver,username,ems_password,file_name)           # -V

# # # ems.ONT_Service_Profile(driver,username,ems_password,file_name)              # V
# # # ems.ONT_QoS_Bandwidth_Profile(driver,username,ems_password,file_name)      # V
# # # ems.ONT_Multicast_Profile(driver,username,ems_password,file_name)          # -V
# # # ems.ONT_UNI_Profile(driver,username,ems_password,file_name)                # -V
# # # ems.ONT_Security_Profile(driver,username,ems_password,file_name)           # V
# # # ems.ONT_ONT_Profile(driver,username,ems_password,file_name)                # -V
# # # ems.ONT_VoIP_SIP_Profile(driver,username,ems_password,file_name)           # -V
# # # ems.ONT_VoIP_Common_Profile(driver,username,ems_password,file_name)        # V
# # # ems.ONT_VoIP_Dial_Plan_Profile(driver,username,ems_password,file_name)     # V
# # # ems.ONT_ACL_Profile(driver,username,ems_password,file_name)                # V 
# # # ems.ONT_Alarm_Profile(driver,username,ems_password,file_name)              # V

# # # ems.ONT_Template_Profile(driver,username,ems_password,file_name)           # X need add create other profile  


# # # ems.Shaping_Profile(driver,username,ems_password,file_name)                        # -V  
# # # ems.Weight_Profile(driver,username,ems_password,file_name)                         # -V
# # # ems.Rate_Limit_Profile(driver,username,ems_password,file_name)                     # -V
# # # ems.GE_Template(driver,username,ems_password,file_name)                            # X  need add create other profile


# print(f"EMS - Profile Test End\n",file = open(file_name, '+a'))

# end_time = time.time() 
# total_time = end_time - start_time
# print(f'Profile Test   Total time: {int(total_time/60/60)} hr {int(total_time/60%60)} min {int(total_time%60)} sec',file = open(file_name, '+a'))
# # # ->>> Out of Memory occured!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# print("[Page]",file = open(file_name, '+a'))


In [ ]:
#    Provision > Service
# ====================================
# ... only set profile no check

# from ems_test_module import *

# Create_ONT_Service_Profile(driver,username,ems_password,file_name)
# Create_ONT_QoS_Bandwidth_Profile(driver,username,ems_password,file_name)
# Create_ONT_Multicast_Profile(driver,username,ems_password,file_name)
# Create_ONT_UNI_Profile(driver,username,ems_password,file_name)
# Create_ONT_ONT_Profile(driver,username,ems_password,file_name)
# Create_ONT_Security_Profile(driver,username,ems_password,file_name)
# Create_ONT_ACL_Profile(driver,username,ems_password,file_name)             
# Create_ONT_Alarm_Profile(driver,username,ems_password,file_name)

# Create_ONT_VoIP_Sip_Profile(driver,username,ems_password,file_name)
# Create_ONT_VoIP_Common_Profile(driver,username,ems_password,file_name)
# Create_ONT_VoIP_Dial_Plan_Profile(driver,username,ems_password,file_name)
# Create_IGMP_Group_Privilege_Profile(driver,username,ems_password,file_name)

# > provision > ont >> service >> +

# mod.Switch_Default(driver)
# time.sleep(1) 
# driver.find_element(By.LINK_TEXT,"Provision").click()
# time.sleep(1)
# driver.find_element("id","service_tab_ont").click() # or driver.find_element(By.LINK_TEXT,"ONT").click()
# time.sleep(1)
# driver.find_element(By.LINK_TEXT,"Service").click()
# time.sleep(1)
# driver.find_element("id","provision_service_create").click()


In [ ]:
# #    Provision > Node Configuration        
# # ====================================

# print("======================================================================================",file = open(file_name, '+a'))
# print("======================================================================================",file = open(file_name, '+a'))
# print("======================================================================================",file = open(file_name, '+a'))
# print(f"EMS - Node Configuration Test Start\n",file = open(file_name, '+a'))


# print("======================================================================================",file = open(file_name, '+a'))
# print("No Card Test",file = open(file_name, '+a'))
# print("======================================================================================",file = open(file_name, '+a'))
# start_time = time.time()
# ems.EMS_test_all_NoCard(driver,url,list1,MSC_card,MSC_slot,msc_port,username,password,file_name,test_retry_num)

# end_time = time.time() 
# total_time = end_time - start_time
# print(f'No Card Test    Total time: {int(total_time/60/60)} hr {int(total_time/60%60)} min {int(total_time%60)} sec',file = open(file_name, '+a'))
# print("[Page]",file = open(file_name, '+a'))


# print("======================================================================================",file = open(file_name, '+a'))
# print("Card Test",file = open(file_name, '+a'))
# print("======================================================================================",file = open(file_name, '+a'))
# start_time = time.time()
# for i in range(list1.__len__()):

#     list_element = list1[i]

#     slot = list_element.split(",")[1]
#     port = list_element.split(",")[2]
#     cardtype = list_element.split(",")[3]
#     print(f"             Now in case:{list_element[:1]},  Slot-Port:{slot}-{port}, {cardtype}",file = open(file_name, '+a'))
#     print("======================================================================================",file = open(file_name, '+a'))
#     ems.EMS_test_all_ACL(driver,url,list_element,MSC_card,MSC_slot,username,password,file_name,test_retry_num) # no MAC, OUI Test !
#     ems.EMS_test_all_Multicast(driver,url,list_element,MSC_card,MSC_slot,username,password,file_name,test_retry_num)
#     ems.EMS_test_all_Switch(driver,url,list_element,MSC_card,MSC_slot,msc_port,username,password,file_name,test_retry_num)
#     ems.EMS_test_all_Port(driver,url,list_element,MSC_card,MSC_slot,msc_port,username,password,file_name,test_retry_num)
#     ems.EMS_test_all_VLAN(driver,url,list_element,MSC_card,MSC_slot,msc_port,username,password,file_name,test_retry_num)
#     print("[Page]",file = open(file_name, '+a'))

# ems.EMS_MSC_Not_Support_Test(driver,username,url,MSC_slot,file_name)

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

# # # ACL
# # ems.EMS_ACL_Anti_MAC_Spoof(driver,url,list1,username,password,file_name)                  # V case
# # ems.EMS_ACL_ARP_Inspection(driver,url,list1,username,password,file_name)                  #   case 
# # ems.EMS_ACL_DHCP_L2Agent(driver,url,MSC_card,username,password,file_name)                 #   msc   => card?
# # ems.EMS_ACL_MAC_Count(driver,url,list1,username,password,file_name)                       #   case
# # ems.EMS_ACL_MAC_Filter(driver,url,list1,username,password,file_name)                      #   case 
# # ems.EMS_ACL_OUI_Filter(driver,url,list1,username,password,file_name)                      #   case  undone
# # ems.EMS_ACL_Packet_Filter(driver,url,list1,username,password,file_name)                   #   case

# # # Multicast
# # ems.Multicast_Port_Mode(driver,url,MSC_card,MSC_slot,username,password,file_name)         # msc need_modify          => card?
# # ems.Multicast_Multicast_MAC(driver,url,list1,username,password,file_name)                 # case
# # ems.Multicast_Multicast_IP(driver,url,list1,username,password,file_name)                  # case

# # # Switch
# # ems.Switch_DSCP(driver,url,list1,username,password,file_name)                               # case  
# # ems.Switch_Storm_Control(driver,url,list1,username,password,file_name)                      # case   

# # # Port
# # ems.Port_GENNI(driver,url,list1,MSC_slot,msc_port,username,password,file_name)                      # case  
# # ems.Port_GE(driver,url,list1,MSC_slot,username,password,file_name)                      # case  
# # ems.Port_Qos(driver,url,list1,username,password,file_name)                           # undone  X   case
# # ems.Port_DHCP(driver,url,list1,username,password,file_name)                     # case

# # # VLAN
# # ems.VLAN_VLAN_Table(driver,url,list1,username,password,file_name)                  # case
# # ems.VLAN_Untag_And_Copy_Cpbit(driver,url,list1,username,password,file_name)        # case

# # # # No Card (173min)
# # ems.EMS_ACL_DHCP_Remote_Backup(driver,url,MSC_card,MSC_slot,username,password,file_name)  #   msc
# # ems.EMS_ACL_PPPoE(driver,url,MSC_card,username,password,file_name)                        #   msc  
# # ems.EMS_ACL_Multicast(driver,url,MSC_card,username,password,file_name)                    #   msc
# # ems.Multicast_IGMP_Setup(driver,url,MSC_card,username,password,file_name)                 # msc need_modify   X
# # ems.Multicast_VLAN_Mode(driver,url,MSC_card,username,password,file_name)                  # msc need_modify   X
# # ems.Multicast_MVLAN_And_MVLAN_Table(driver,url,MSC_card,username,password,file_name)      # msc
# # # (Sys)                                                                                     # msc                                                                               
# # ems.Sys_General_Setup(driver,url,MSC_card,username,password,file_name) 
# # Ip_setup_gateway = ems.Sys_IPv4_Setup(driver,url,MSC_card,username,password,file_name)
# # ems.Sys_IPv6_Setup(driver,url,MSC_card,username,password,file_name)
# # ems.Sys_IP_Route(driver,url,MSC_card,username,password,file_name, Ip_setup_gateway)    
# # ems.Switch_General(driver,url,MSC_card,username,password,file_name)                         # msc 
# # ems.Switch_Dot3ad(driver,url,MSC_card,MSC_slot,msc_port,username,password,file_name)        # msc   undone
# # ems.Switch_Isolation(driver,url,MSC_card,username,password,file_name)                       # msc 
# # ems.VLAN_Tag_Mode_Setup(driver,url,MSC_card,MSC_slot,username,password,file_name)         # msc



# end_time = time.time() 
# total_time = end_time - start_time
# print(f'Card Test + MSC Test   Total time: {int(total_time/60/60)} hr {int(total_time/60%60)} min {int(total_time%60)} sec',file = open(file_name, '+a'))



In [ ]:
#                End
# ====================================== 

end_time = time.time() 
total_time = end_time - start_time0
print(f'Total time: {int(total_time/60/60)} hr {int(total_time/60%60)} min {int(total_time%60)} sec',file = open(file_name, '+a'))

# test_report.close()

driver.quit()